In [14]:
import os
import json
import random
import pandas as pd
import shutil
import uuid
import splitfolders
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import albumentations as A
import cv2
import os
import utils

In [15]:
random.seed(42)

In [16]:
DATA_PATH = "../../data/"
TRAIN_CSV_PATH =  DATA_PATH + "raw/train_label_to_category.csv"
EXTRA_IMAGES_DIR =  DATA_PATH + "landmark_vietnam_dataset"
SLIPT_IMAGES_DIR = DATA_PATH + "landmark_vietnam_dataset_split"
FIX_IMAGES_DIR = DATA_PATH + "newdata"
OUTPUT_CSV = DATA_PATH + "processed/additional_train.csv"

## Syncing folder

In [18]:
utils.parallel_sync_folders(EXTRA_IMAGES_DIR, FIX_IMAGES_DIR)

Syncing: 100%|██████████| 104/104 [00:01<00:00, 88.11it/s]


Successfully synced 104/104 folders


## Resize and Rename image

In [19]:
utils.parallel_resize_dataset(EXTRA_IMAGES_DIR)

Resizing Images: 100%|██████████| 2217/2217 [00:03<00:00, 557.77it/s]


Successfully resize 2217/2217 images


## Clean trash files

In [21]:
utils.clean_non_jpg_files(EXTRA_IMAGES_DIR)

Cleaning files: 100%|██████████| 2450/2450 [00:00<00:00, 65121.51it/s]


Done: deleted 233 non-.jpg files.


In [22]:
def count_images_in_big_folder(folder_path):
    total_images = 0
    # Danh sách các đuôi ảnh bà muốn đếm
    valid_extensions = ('.jpg', '.jpeg', '.png', '.webp')
    
    for root, dirs, files in os.walk(folder_path):
        # Đếm các file có đuôi nằm trong danh sách valid_extensions
        image_files = [f for f in files if f.lower().endswith(valid_extensions)]
        total_images += len(image_files)
        
    return total_images

utils.count_images_in_folder(EXTRA_IMAGES_DIR)

2217

## Split dataset before Augmentation

In [23]:
input_folder = EXTRA_IMAGES_DIR
output_folder = SLIPT_IMAGES_DIR

# Split 70:20:10
if not os.path.exists(output_folder):
    splitfolders.ratio(input_folder, output=output_folder, seed=42, ratio=(.7, .2, .1))

Copying files: 2217 files [00:03, 671.95 files/s] 


In [24]:
def check_split_length(base_path):
    print(f"{'Split':<10} | {'Total Images':<15}")
    print("-" * 30)
    
    for split in ['train', 'val', 'test']:
        split_dir = os.path.join(base_path, split)
        if not os.path.exists(split_dir):
            print(f"{split:<10} | Folder not found!")
            continue
            
        # Đếm tất cả file ảnh trong tất cả các folder con
        count = sum([len(files) for r, d, files in os.walk(split_dir) 
                     if any(f.lower().endswith(('.jpg', '.jpeg', '.png')) for f in files)])
        
        print(f"{split.upper():<10} | {count:<15}")

check_split_length(SLIPT_IMAGES_DIR)

Split      | Total Images   
------------------------------
TRAIN      | 1463           
VAL        | 359            
TEST       | 395            


## Data Augmentation

In [26]:
utils.data_augmentation_parallel(src_dir=SLIPT_IMAGES_DIR+"/train", num_variants=2)

Data Augmentation processing: 1463 images...


100%|██████████| 1463/1463 [00:10<00:00, 139.95it/s]

Original images: 1463
Created variants: 2926
Total images: 4389
Done Data Augmentation


## Prepare metadata for JSONL file

In [27]:
# Prepare data mapping to get landmark_name from root file
df_mapping = pd.read_csv(TRAIN_CSV_PATH, usecols=['landmark_id', 'category'])
df_mapping = df_mapping.drop_duplicates(subset=['landmark_id'])

df_mapping['landmark_id'] = df_mapping['landmark_id'].astype(str)

mapping_dict = df_mapping.set_index('landmark_id').to_dict('index')

In [29]:
split_names = ['train', 'val', 'test']
all_image_metadata = []

for name in split_names:
    print(f"Processing folder {name}")
    src_path = os.path.join(SLIPT_IMAGES_DIR, name)
    df_split, global_landmark_info = utils.create_metadata(src_path, name, mapping_dict=mapping_dict)
    
    df_split.to_csv(f"{DATA_PATH}processed/{name}_metadata.csv", index=False, encoding='utf-8-sig')
    
    all_image_metadata.append(df_split)

Processing folder train
Processing folder val
Processing folder test


In [30]:
df_landmark_final = pd.DataFrame(global_landmark_info)
df_landmark_final.to_csv(DATA_PATH + "processed/landmark_vietnam_info.csv", index=False, encoding='utf-8-sig')

# All train, test, val
pd.concat(all_image_metadata).to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

## Create JSONL File

In [31]:
split_names = ['train', 'val', 'test']

df = pd.read_csv(DATA_PATH + "processed/landmark_vietnam_info.csv")

for name in split_names:
    src_dir = SLIPT_IMAGES_DIR + "/" + name
    utils.create_model_dataset(df=df, file_output= DATA_PATH + f"{name}_vietnam_landmark.jsonl", dataset_dir=src_dir)

Successfully create 4389 QA pairs
Successfully create 359 QA pairs
Successfully create 395 QA pairs
